# 出席管理・授業後アンケート v2 — Colab 共有用ノート

このノートを上から実行すると、**4画面（生徒用 / 校舎スタッフ用 / 講師・教室担当用 / 校舎QR読取タブレット）を、それぞれ別ウィンドウで開くリンク**を発行します。実データ（CSV）を読み込むので、出席登録・アンケート・各マスタなどの機能を実際に操作して確認できます。

**共有の流れ（要約）**
1. GitHub に最新を push
2. リポジトリを Public にする（最も簡単）／または閲覧者に権限付与
3. このノートを Colab で開く →「ランタイム > すべてのセルを実行」
4. 表示された4つのリンクから各画面を別ウィンドウで開く
5. 「ファイル > ドライブにコピーを保存」→「共有」でノートのリンクを発行し、相手に送る

注意: Colab 上の画面操作はブラウザ内の一時状態です（各自の画面内でのみ反映、「初期データに戻す」で復元可）。

## 共有までに必要な作業（チェックリスト）

### A. リポジトリ準備（PC側・1回）
- [ ] 変更を commit して GitHub に push（`git push`）
- [ ] リポジトリを **Public** にする: GitHub → Settings → 最下部 Danger Zone → Change repository visibility → Public
  - private のままにしたい場合は末尾「private のまま使う場合」セルを使用

### B. Colab で開いて実行
- [ ] https://colab.research.google.com →「ファイル > ノートブックを開く > GitHub」で `YutzMame/tokushin_app_mock` → `mock_app_v2/colab_share_v2.ipynb`
- [ ] 「ランタイム > すべてのセルを実行」→ 発行された4リンクを開く

### C. 共有
- [ ] 「ファイル > ドライブにコピーを保存」→ 右上「共有」→「リンクを知っている全員（閲覧者）」→ リンクを送付

## セットアップ（最初に実行）

リポジトリを取得し、表示ヘルパーを読み込みます。Public リポジトリならそのまま成功します。

In [ ]:
# === セットアップ: リポジトリ取得 + 表示ヘルパー読込（最初に実行）===
import importlib.util, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YutzMame/tokushin_app_mock.git"
REPO_DIR = "tokushin_app_mock"

def find_helper():
    for base in (".", REPO_DIR, f"/content/{REPO_DIR}"):
        p = Path(base) / "mock_app_v2" / "colab_preview_v2.py"
        if p.exists():
            return p
    return None

helper = find_helper()
if helper is None:
    print("リポジトリを取得します...")
    res = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    print(res.stdout or "", res.stderr or "")
    helper = find_helper()

if helper is None:
    raise SystemExit(
        "リポジトリを取得できませんでした。\n"
        "・private の場合: GitHub で Public にする（最も簡単）か、末尾のトークン方式セルを使ってください。\n"
        "・既にローカルにある場合: リポジトリのルートでこのノートを開いてください。"
    )

os.chdir(helper.resolve().parents[1])  # リポジトリのルートへ
spec = importlib.util.spec_from_file_location("colab_preview_v2", "mock_app_v2/colab_preview_v2.py")
preview_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preview_v2)
print("OK: 表示ヘルパーを読み込みました。下のセルを実行してください。")


## 4画面のリンクを発行（CSV込み・別ウィンドウ）

下のセルを実行すると、4画面それぞれの「別ウィンドウで開く」リンクが表示されます。実データCSVを読み込むため、機能を実際に操作できます。

（Colab以外・ローカルで実行した場合は `http://localhost:8000/...` のURLが表示されます。）

In [ ]:
httpd = preview_v2.serve_links_v2()  # 生徒用 / 校舎用 / 講師用 / タブレット の4リンクを発行

## （private のまま使う場合）トークンでクローン

リポジトリを Public にしない場合のみ、下のセルでトークンを使ってクローンします。共有相手にもリポジトリ閲覧権限と各自のトークンが必要になるため、関係者だけで使う想定です。

In [ ]:
# === （private のまま使う場合のみ）Personal Access Token でクローン ===
# GitHub の Settings > Developer settings > Personal access tokens で repo 読取権限のトークンを発行。
# 実行するとトークン入力欄が出ます（ノートには保存されません）。実行後、上のセットアップセルを再実行。
import getpass, subprocess, os
token = getpass.getpass("GitHub Personal Access Token: ")
subprocess.run(["git", "clone", f"https://{token}@github.com/YutzMame/tokushin_app_mock.git"])
if os.path.isdir("tokushin_app_mock"):
    os.chdir("tokushin_app_mock")
print("クローン完了したら、上のセットアップセルをもう一度実行してください。")
